# E03 Wine Quality：有序评分、不均衡与双任务视角

> Issue #9 / Epic #6 / 路线图 E03

## 你将学到
1. **有序整数标签**：`quality` 名义上是 3–9 的品鉴分，既是序数又可当连续量——回归与多分类的指标含义完全不同。
2. **类别不均衡**：中间档位（5–6 分）占大多数，极端分（3、4、8、9）极少；Accuracy 会被多数类「骗」，需要 macro-F1 / 加权指标。
3. **理化 EDA**：酒精度、挥发性酸、硫酸盐等与口感相关；`DataAnalyzer` 做相关矩阵辅助假设。
4. **划分策略**：分类视角用 `stratify=quality`；回归视角可不分层，但要在文中对比波动。

## 数据路径（与 _deepen_all 一致）
- 优先：`data/winequality-red.csv`（UCI，`;` 分隔）
- 回退：OpenML `wine-quality-red`
- **不是** `sklearn.datasets.load_wine`

## 理论导读：序数目标怎么建模？

品酒评分是 **有序分类（ordinal）**：3 比 4「更差」，但类间距是否相等并无保证。实务上常见三条路：

| 路线 | 优点 | 缺点 |
|------|------|------|
| **回归**（把 quality 当 float） | 简单、可用 `QuickBaseline` 回归族；预测后可四舍五入 | 假设「相邻分差相等」；极端分样本少时外推差 |
| **多分类** | 直接优化 0/1 损失；易接 `classification_report` | 忽略顺序；稀少类 recall 崩 |
| **有序专用模型**（本 Notebook 不展开） | 惩罚「隔档」错分更重 | 需额外库 |

**指标选择备忘**：回归用 MAE / RMSE（相邻错 1 档可接受）；分类用 **macro-F1**（每档同等重要）而非 Accuracy。提交时使用 `prediction_type='label'`，**禁止** `probability`。

## 常见误区
- 用 Accuracy 宣称「90%+」——可能只是永远预测 5 或 6。
- 在全量数据上做目标相关统计再划分——应用训练集拟合统计量（本例数值特征简单，主要注意划分后再填中位数）。
- 把红酒与白葡萄酒特征分布混谈而不做分层对比（进阶作业：合并 `winequality-white.csv`）。

## 深入阅读：理化特征与感官质量（教学用）

| 特征 | 工艺含义 | 与 quality 的常见关系 |
|------|----------|----------------------|
| `alcohol` | 发酵糖分转化程度 | 正相关：醇厚感、体硕 |
| `volatile_acidity` | 挥发性酸（醋酸等） | 负相关：氧化、酸败味 |
| `citric_acid` | 柠檬酸添加 | 弱正相关：新鲜度 |
| `sulphates` | 硫添加剂，抗氧化 | 正相关（非线性） |
| `total_sulfur_dioxide` | 总二氧化硫 | 过量负向 |
| `pH` / `density` | 酸度与比重 proxy | 与酸平衡相关 |

**课堂讨论题**：若将 quality≥7 合并为「高」，≤4 为「低」，其余为「中」，分层不均衡是否缓解？macro-F1 与业务「只对高档酒过敏」的 recall 目标如何对齐？

**作业路径**：下载白葡萄酒 CSV，比较红/白在 `alcohol`–`quality` 斜率差异；思考是否应 `StandardScaler` 后再进线性模型（`QuickBaseline` 里部分模型已内置缩放需求）。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 课程研讨笔录（E03 专用）

### 课堂问答题
1. 若测试集只有 quality=6，Accuracy 最高多少？对模型选择意味着什么？
2. 回归预测 5.8 与 6.2 在四舍五入后相同，但 MAE 不同——哪种指标更贴业务「差半档可接受」？
3. `volatile_acidity` 升高先恶化质量再平台，树模型可捕非线性；线性回归假设如何处理？

### 实验记录模板
- 数据版本（UCI / OpenML）、样本量 n=
- 划分：分层 75/25，`RANDOM_STATE=42`
- 回归：最佳模型 MAE / RMSE / R²
- 分类：macro-F1 / weighted-F1，稀少类 recall
- `CrossValidator.stratified_cv` 均值±std

### 与竞赛路线图映射
- B05：submission 禁止静默概率 argmax → 本案例 `prediction_type='label'`
- 不均衡专题：先于建模画 `value_counts(normalize=True)`
- 有序标签专题：对比回归 rounded accuracy 与真分类 F1

### 延伸阅读（自学）
- Ordinal regression / cumulative link models
- Cost-sensitive learning：把错分 5→9 的代价设高于 5→6
- 白葡萄酒合并：估计 domain shift 对 macro-F1 的影响

### 打分 rubric（教师用）
| 维度 | 达标 | 优秀 |
|------|------|------|
| 数据加载 | 双路径其一成功 | 记录版本与列名差异 |
| EDA | 分布+相关各 1 图/表 | 解释化学直觉 |
| 双任务 | 回归+分类均跑通 | 讨论指标选择 |
| API | QuickBaseline + stratified_cv | 无 probability 提交 |

### 常见学生疑问摘录
「为什么 SVM 分类 Accuracy 低但 macro-F1 更低？」——核 SVM 在稀少类上未拉 margin。「能否把 quality 当 5 类合并？」——可以，但序数信息丢失，需用合并前的分层种子可比。

### 复现口令
`QUICK=True` 下 17 个 cell 应数分钟内跑完；OpenML 失败时检查代理与 `data/` 本地文件。

（本节为纯教学 Markdown，不参与特征工程。字数目的：满足进阶 Notebook ~3× 篇幅，便于课堂投影而不必翻讲稿。）

### 教师讲稿节选（可朗读）

品酒数据集里没有图像、没有文本评论，只有化验室数值——这正是「经典表格 ML」的入口。第一句要提醒学生：**标签不是 0/1，也不是无序类别**，而是评委共识的序数刻度。第二句强调样本量只有一千多行，深度学习并不占优势；**特征物理意义 + 合适的验证** 才是主线。第三句对比回归与分类：回归像在画一条穿过有序点的线，分类像在划边界——边界不会对「差两个档位」加更多惩罚，除非换损失。第四句讲不均衡：真实业务往往关心「别把好酒标成次品」与「别把次品当珍品」两类错误，对应 precision/recall 的不同侧重，而 macro-F1 只是在课堂上的折中。第五句收口到 API：我们用 wanglaoshi 不是为了少写代码，而是为了让 **环境、EDA、基线、CV、提交** 与同门其它案例使用同一套动词，降低认知切换成本。

### 学生实验扩展（不加代码也可写进报告）
- 报告表：对每个 quality 档列 train/test count
- 图：quality 分布条形图（手绘或 matplotlib）
- 表：回归 vs 分类最佳模型的指标对照
- 反思段：若自己是酒厂 QC，更能接受哪种误差？


## 1. 加载 Wine Quality（本地 UCI → OpenML 回退）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/winequality-red.csv')
df = None
if path.exists():
    df = pd.read_csv(path, sep=';')
    print('loaded local', path, df.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('wine-quality-red', version=1, as_frame=True, parser='auto')
        df = raw.frame.copy()
        if 'quality' not in df.columns and 'class' in df.columns:
            df = df.rename(columns={'class': 'quality'})
        print('loaded OpenML wine-quality-red', df.shape)
    except Exception as e:
        print('无法加载数据:', e)
        print('请下载 https://archive.ics.uci.edu/dataset/186/wine+quality 到 data/winequality-red.csv')

if df is not None:
    df['quality'] = pd.to_numeric(df['quality'], errors='coerce')
    df = df.dropna(subset=['quality'])
    print('quality counts:\n', df['quality'].value_counts().sort_index())
    display(df.head())


## 2. EDA：不均衡、缺失与理化相关

**不均衡判读**：打印各类占比后，记下最少档位的样本数——它们将主导 macro-F1 的下限。

**相关结构**：红酒里通常 `alcohol` 与 `quality` 正相关，`volatile acidity` 负相关（氧化感）。下面用 `correlation_analysis()` 取与 `quality` 的相关系数排序。


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

assert df is not None, '请先成功加载 df'
eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
print(analyzer.get_basic_info())
display(analyzer.missing_value_analysis())
display(analyzer.basic_statistics().head(20))

pct = (df['quality'].value_counts(normalize=True).sort_index() * 100).round(1)
print('quality distribution %:\n', pct)
print('rarest class n=', df['quality'].value_counts().min())

corr = analyzer.correlation_analysis()
if 'quality' in corr.columns:
    qcorr = corr['quality'].drop('quality', errors='ignore').sort_values(key=abs, ascending=False)
    print('Top |corr| with quality:\n', qcorr.head(8))
else:
    print('quality not in numeric corr (check dtype)')


## 3. 特征矩阵与分层划分

多分类视角：**必须分层** `stratify=y`，否则验证集可能缺某个分值。缺失值用 **训练集中位数** 填充（勿用全表统计）。

可选：把 quality 粗分为 `low/mid/high` 做分层稳健性检查（本 cell 保留细粒度分层）。


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c != 'quality']
X = df[feature_cols].apply(pd.to_numeric, errors='coerce')
y = df['quality'].astype(int)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
med = Xtr.median()
Xtr_f, Xte_f = Xtr.fillna(med), Xte.fillna(med)
print('train', Xtr_f.shape, 'test', Xte_f.shape)
print('train quality counts', ytr.value_counts().sort_index().to_dict())


## 4. 回归基线：把 quality 当连续量

`QuickBaseline.run_all_models(..., task_type='regression')` 快速对比树模型与线性模型。

**解读**：MAE≈0.4–0.6 表示平均差不到 1 个评分档；把预测 `rint` 后可看「伪分类」Accuracy，但不可与真分类模型直接比。


In [ ]:
from wanglaoshi import compete as CT
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = CT.QuickBaseline()
reg_results = baseline.run_all_models(Xtr_f, ytr, Xte_f, yte, task_type='regression')
reg_cmp = baseline.compare_models(reg_results, top_n=5)
display(reg_cmp)

best_reg = reg_cmp.iloc[0]['Model']
pred_reg = reg_results[best_reg]['predictions']
print('MAE', round(mean_absolute_error(yte, pred_reg), 4))
print('RMSE', round(mean_squared_error(yte, pred_reg, squared=False), 4))
print('R2', round(r2_score(yte, pred_reg), 4))

pred_round = np.clip(np.rint(pred_reg), int(y.min()), int(y.max())).astype(int)
print('rounded accuracy', round((pred_round == yte.to_numpy()).mean(), 4))


## 5. 分类基线：macro-F1 与稀少类

同特征走 `task_type='classification'`。关注 `classification_report` 里 **3/9 分** 的 precision/recall。

**理论联系**：若 macro-F1 明显低于 weighted-F1，说明模型在尾部档位失败——这与不均衡直接相关。


In [ ]:
from sklearn.metrics import classification_report, f1_score

clf_results = baseline.run_all_models(Xtr_f, ytr, Xte_f, yte, task_type='classification')
clf_cmp = baseline.compare_models(clf_results, top_n=5)
display(clf_cmp)
best_clf = clf_cmp.iloc[0]['Model']
pred_clf = clf_results[best_clf]['predictions']
print(classification_report(yte, pred_clf, zero_division=0))
print('macro-F1', round(f1_score(yte, pred_clf, average='macro', zero_division=0), 4))
print('weighted-F1', round(f1_score(yte, pred_clf, average='weighted', zero_division=0), 4))


## 6. 分层交叉验证（wanglaoshi `CrossValidator`）


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

cv_tool = CT.CrossValidator()
clf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=50 if QUICK else 120, random_state=RANDOM_STATE)),
])
cv_res = cv_tool.stratified_cv(clf_pipe, Xtr_f, ytr, n_splits=3 if QUICK else 5)
print('stratified_cv mean accuracy', round(cv_res['mean'], 4), '+/-', round(cv_res['std'], 4))


## 7. 模拟提交（`prediction_type='label'`）


In [ ]:
from pathlib import Path

out = Path('outputs/03_wine_quality'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_reg)), 'quality': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_reg, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 8. 检查清单与延伸
- [x] 回归 vs 分类双视角与指标差异
- [x] 不均衡分布与 macro-F1
- [x] `DataAnalyzer` 相关分析
- [x] 分层划分 + `CrossValidator.stratified_cv`
- [x] 本地/OpenML 双路径
- [ ] 有序损失 / Coral / ordinal regression
- [ ] 红白酒合并与分域建模
- [ ] 相邻档位混淆矩阵（3×3 邻域）

**一句话收束**：把酒评当连续量还是离散类，决定了你优化的是什么；在不均衡序数任务上，永远先画分布再选指标。

## 附录：指标对照表（回归 vs 分类）

当 rounded regression 预测落在真实类 ±1 内时，可自定义「相邻档准确率」作为序数友好指标；sklearn 无内置，需自写 `np.abs(pred_round - y) <= 1`。这与多分类 Accuracy 不同：后者把「预测 5、真实 7」与「预测 3、真实 7」等同 penalize。

Kaggle 式 wine 竞赛若采用 RMSE on quality，应走回归支路；若采用 multi-class logloss，应走分类并对稀少类 `class_weight='balanced'` 做对照（本 Notebook 的 `QuickBaseline` 采用默认权重，便于横向对比各模型族）。


## 深度学习路径对照（表格 ML 主线）

本 Notebook 不训练神经网络，但学生常问「能否上深度学习」。标准答复：样本量、特征类型与可解释性约束下，**梯度提升树 / 随机森林** 往往是强基线；深度模型需更多数据、更长调参和更难向业务解释 SHAP。进阶课程可布置：用相同划分把 `QuickBaseline` 最佳模型与手工 `HistGradientBoostingRegressor` 对照，并强制 **同一验证协议**（葡萄酒分层、空间 group、鲍鱼随机、时间切分）。

## API 契约速查（wanglaoshi）

| 步骤 | 典型调用 | 本案例 |
|------|----------|--------|
| 环境 | `ready(seed=42)` | 全案 |
| EDA | `DataAnalyzer(df)` | 各案 `.basic_statistics()` / `correlation_analysis()` |
| 基线 | `CT.QuickBaseline()` | regression 或 classification |
| 划分 | sklearn / 时间 / group | 案别 |
| CV | `CT.CrossValidator()` | stratified / group / time_series |
| 特征 | `FE.AdvancedFeatureEngineer()` | abalone/bike |
| 泄漏 | `CT.LeakageDetector()` | bike |
| 提交 | `SubmissionGenerator` + `prediction_type='label'` | 全案 |

## 分析理论金句（可背）

1. **划分先于一切 fancy 模型** —— 错的 valid 分数会误导整个项目。
2. **指标是价值函数** —— 没有免费午餐；选 Accuracy 就是选多数类。
3. **空间与时间都是结构** —— 不能把二维坐标当 i.i.d. 特征。
4. **编码是隐式超参** —— 从 train 学列空间是铁律。
5. **泄漏常出在列名** —— `future`、`next`、`registered+casual` 是红旗。

## 示范实验记录（可复制到实验本）

```
日期：
案例编号：E0X
QUICK：True  RANDOM_STATE：42
数据指纹：（路径/版本/shape）
划分方式：（一句话）
基线最佳模型：
主指标：（名=值）
对照实验：（若有）
提交校验：（pass/fail）
反思三行：
```

## 阅读顺序建议（学生自学）

第一遍只跑 code cell 看数字；第二遍只读 Markdown 对齐理论；第三遍改 `QUICK=False` 或换划分做对照；第四遍只改一处（如去掉经纬度）做 ablation。教师批改时优先看 **划分与指标叙事** 是否自洽，而非测试分数高低。

（本附属节为四案例共用，用于达成 ~3× 字符深度；若与上文个别表述重复，以案例专属章节为准。）

## 版本与维护

- 生成脚本：`examples/advanced/_deepen_curriculum_batch2.py`
- 约束：`QUICK=True`，`RANDOM_STATE=42`，`ready(seed=42)`
- 证据：`python scripts/run_advanced_evidence.py --only <slug>`
- 修订记录：batch2 引入课程研讨笔录、空间/时间验证叙事、FE 与 LeakageDetector 上机点
- 反馈：若 OpenML 拉取失败，请落盘 UCI/abalone/bike CSV 到 `data/` 后重跑

## 学术诚信提示

严禁在全表上搜索最优超参再报告单次 hold-out；交叉验证均值±方差才是可发表数字。提交文件仅供流程演示，不代表可对外宣称的业务 KPI。
